In [2]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import timm
from PIL import Image
%load_ext autoreload
%autoreload 2

# hyperparameters
device = torch.device("cpu")

# initialize model
model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0)

# load state dict containing miscellaneous state or just the model weights
state_dict = torch.load("../weights/MegaD_elephant_w.pt", map_location=device)

if "optimizer" in state_dict:
    model.load_state_dict(state_dict["model"])
else:
    model.load_state_dict(state_dict)

model.to(device)
model.eval()

/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_3807022/2207306945.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend y

SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (layers): Sequential(
    (0): SwinTransformerStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): SwinTransformerBlock(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): WindowAttention(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (softmax): Softmax(dim=-1)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU(approximate='none')
            (drop1): 

In [14]:
# Make a dataloader that mixes wevrything randomly loads the images splits a train and test and computes the embeddings
from data_handler import EleHandler
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from pathlib import Path
import torch
import torch.nn.functional as F
from seek_code import SEEK



# Create dataset and dataloaders
dataset = EleHandler(dataset_dir=Path('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse'),
                       dictonary_path=Path('/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary.csv'),
                       transform='MegaDescriptor-elephant'
                       )


train_size = int(0.7 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=128, sampler=RandomSampler(train_dataset))
val_loader = DataLoader(val_dataset, batch_size=128, sampler=RandomSampler(val_dataset))
test_loader = DataLoader(test_dataset, batch_size=128, sampler=RandomSampler(test_dataset))

model.to('cuda')
layer = torch.nn.Linear(768, 63).to('cuda')

In [16]:
def closest_valid_one_hot(prob_vector):
    """
    Converts a probability tensor into the closest valid one-hot encoded representation for each attribute.
    """
    closest_one_hot = []
    index = 0

    for group in SEEK.attribute_names:
        slice_length = SEEK.lengths[group]
        prob_slice = prob_vector[:, index:index + slice_length]
        one_hot_slice = torch.zeros_like(prob_slice)
        
        max_indices = torch.argmax(prob_slice, dim=1)
        one_hot_slice[torch.arange(prob_slice.size(0)), max_indices] = 1.0
        closest_one_hot.append(one_hot_slice)
        
        index += slice_length

    return torch.cat(closest_one_hot, dim=1)

def calculate_attribute_accuracy(predicted, labels):
    """
    Calculates the accuracy per attribute based on the closest valid one-hot encoded vectors
    of predicted logits and true labels, with averaging for specified pairs.
    Also computes the accuracy for the entire SEEK code, counting correct only when all attributes match.
    """
    # Convert predicted logits to closest valid one-hot representation
    predicted_one_hot = closest_valid_one_hot(predicted)
    
    # Initialize accuracy tracking for individual and averaged attributes
    accuracies = {name: 0 for name in SEEK.attribute_names}
    index = 0

    # Define pairs to be averaged
    average_pairs = {
        'tusk': ['right_tusk', 'left_tusk'],
        'tear_1': ['R_tear_1', 'L_tear_1'],
        'hole_1': ['R_hole_1', 'L_hole_1'],
        'tear_2': ['R_tear_2', 'L_tear_2'],
        'hole_2': ['R_hole_2', 'L_hole_2'],
        'extreme': ['right_extreme', 'left_extreme']
    }

    # Track individual accuracies
    individual_accuracies = {name: 0 for name in SEEK.attribute_names}
    batch_count = predicted.size(0)

    # Initialize counter for whole SEEK code accuracy
    correct_whole_code = 0

    for name in SEEK.attribute_names:
        length = SEEK.lengths[name]
        pred_slice = predicted_one_hot[:, index:index + length]
        label_slice = labels[:, index:index + length]

        # Calculate per-sample accuracy
        pred_indices = torch.argmax(pred_slice, dim=1)
        label_indices = torch.argmax(label_slice, dim=1)
        individual_accuracies[name] = (pred_indices == label_indices).float().mean().item()

        index += length

    # Calculate whole SEEK code accuracy
    correct_whole_code = torch.all(predicted_one_hot == labels, dim=1).float().mean().item()

    # Calculate and store averaged accuracies for paired attributes
    for key, pair in average_pairs.items():
        averaged_accuracy = (individual_accuracies[pair[0]] + individual_accuracies[pair[1]]) / 2
        accuracies[key] = averaged_accuracy

    # Include remaining unpaired attributes in accuracies
    for name in SEEK.attribute_names:
        if name not in [p for sublist in average_pairs.values() for p in sublist]:
            accuracies[name] = individual_accuracies[name]

    # Add whole SEEK code accuracy to the results
    accuracies['whole_code_correct'] = correct_whole_code

    return accuracies

In [17]:
optimizer = torch.optim.Adam(layer.parameters(), lr=0.001)

def epoch_pass(epoch, loader, training=True):
    model.eval()  # Freeze model for inference
    layer.train() if training else layer.eval()

    total_loss = 0
    total_accuracies = {name: 0 for name in SEEK.attribute_names}
    num_batches = len(loader)

    for batch_index, batch in enumerate(loader):
        images, subject_id, ele_id, identified, subject_SEEK, ele_SEEK = batch

        # Prepare labels and transfer them to GPU
        labels = torch.stack([SEEK(s).one_hot_encode() for s in subject_SEEK]).to('cuda')
        images = images.to('cuda')

        # Obtain embeddings from the frozen model
        with torch.no_grad():
            embeddings = model(images)

        # Forward pass through the trainable layer
        outputs = layer(embeddings)

        # Calculate loss directly between outputs and labels
        loss = F.mse_loss(outputs, labels)
        total_loss += loss.item()

        # Calculate accuracy for each attribute
        predicted_SEEK = closest_valid_one_hot(outputs)
        batch_accuracies = calculate_attribute_accuracy(predicted_SEEK, labels)
        for name in SEEK.attribute_names:
            total_accuracies[name] += batch_accuracies[name]  # Accumulate batch accuracy

        # Print batch loss and accuracies
        print(f"Epoch {epoch}, Batch {batch_index + 1}, Loss: {loss.item()}")
        for name, accuracy in batch_accuracies.items():
            print(f"  {name} Accuracy: {accuracy * 100:.2f}%")

        # Backpropagation only for the linear layer if training
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Calculate epoch-level average accuracies
    average_loss = total_loss / num_batches
    average_accuracies = {name: total_accuracies[name] / num_batches for name in SEEK.attribute_names}

    # Print epoch-level accuracies
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Average Loss: {average_loss}")
    for name, accuracy in average_accuracies.items():
        print(f"  {name} Epoch Accuracy: {accuracy * 100:.2f}%")
    
    return average_loss, average_accuracies

In [ ]:
for epoch in range(3):
    train_loss, train_acc = epoch_pass(epoch, train_loader, training = True)
    val_loss, val_acc = epoch_pass(epoch, val_loader, training = False)